<a href="https://colab.research.google.com/github/dacode100/Sistemas-operativos/blob/colab-sistemas/Daniel_Ramirez_Practica_Sistemas_Operativos_U2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Administración de Procesos - Jearaquía, Implantación, IPC
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad II:** Administración de Procesos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Examinar la tabla de procesos del sistema operativo huésped utilizando comandos nativos de Linux (`ps`) en Google Colab para identificar atributos clave como identificadores de procesos (**PID**/**PPID**) y relaciones jerárquicas.

2. Explorar la estructura de datos interna que el kernel de Linux expone a través del sistema de archivos virtual (`/proc`) para comprender cómo se recopila y expone la información de ejecución a nivel de sistema.

3. Analizar la jerarquía de procesos y las relaciones de descendencia en un entorno Linux real empleando herramientas como pstree y el sistema de archivos virtual `/proc` en Google Colab.

4. Implementar las llamadas al sistema clásicas de control de procesos (`fork`, `exec`, `exit`, `wait`) mediante un simulador orientado a objetos en C++17.

5. Examinar fenómenos fundamentales de los sistemas operativos, tales como la persistencia de metadatos en estados Zombie y la adopción automática de procesos huérfanos por parte del proceso raíz (`init`).

6. Examinar los recursos de IPC activos en el sistema operativo huésped (**memoria compartida, semáforos y colas de mensajes**) mediante el comando `ipcs` y parámetros del kernel en `/proc`.

7. Implementar el control de concurrencia y la exclusión mutua aplicando **semáforos** ***POSIX*** (`sem_t`, `sem_wait`, `sem_post`) para resolver el problema clásico del **Productor-Consumidor** con un buffer circular.

8. Desarrollar canales de comunicación unidireccional entre procesos independientes utilizando las llamadas al sistema `pipe()`, `fork()`, `read()` y `write()`.

---

## 2. Sección Práctica 1: Inspección del Sistema Operativo.

En esta sección analizarás los fundamentos de la administración de procesos en los sistemas operativos modernos mediante la inspección de la tabla de procesos en un entorno Linux real(Servidor Linux provisto por Google).

### [CÓDIGO 2.1] inspeccionar la tabla de procesos real del sistema operativo huésped
```bash
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15
```

In [1]:
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0    984   560 ?        Ss   23:38   0:00 /sbin/docker-
root           7  0.4  0.5 1193484 70564 ?       Sl   23:38   0:04 /tools/node/b
root           9  0.0  0.0   7344  3448 ?        S    23:38   0:00 /bin/bash -e 
root          11  0.0  0.0   7344  2008 ?        S    23:38   0:00 /bin/bash -e 
root          13  0.0  0.2 1286680 33604 ?       Sl   23:38   0:00 /usr/colab/bi
root          43  0.0  0.0   5720  2096 ?        Ss   23:38   0:00 tail -n +0 -F
root          49  0.0  0.0   5720  2208 ?        Ss   23:38   0:00 tail -n +0 -F
root          75  2.4  0.0      0     0 ?        Z    23:38   0:22 [python3] <de
root          76  0.3  0.5 100932 77784 ?        S    23:38   0:03 python3 /usr/
root          89  0.0  0.0   7456  2124 ?        S    23:38   0:00 /bin/bash -e 
root          90  0.0  0.0  12024  7748 ?        S    23:38   0:00 sshd: /usr/sb
root         106  0.8  1.2 665316 

### [CÓDIGO 2.2] Implementación de la Estructura struct BCP en C++17
```cpp
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <string>
#include <vector>
#include <iomanip>
#include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}
```

In [2]:
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <string>
# include <vector>
# include <iomanip>
# include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}

Writing bcp_struct.cpp


### [CÓDIGO 2.3] Compilación y ejecución del código
```bash
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct
```

In [3]:
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct

   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   101) - Program: chrome.exe          │
├-----------------------------------------------------------------┤
│ Estado Actual  : EJECUTANDO      | PPID Padre     : 1          │
│ Contexto CPU   : PC=0x4200 | SP=0xFFF0 | AC=150    │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 5          │
│ Memoria Asign. : Base=0x100000 | Límite=655360 Bytes  │
└-----------------------------------------------------------------┘

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID: 10200) - Program: g++_compiler00000000│
├-----------------------------------------------------------------┤
│ Estado Actual  : BLOQUEADO000000 | PPID Padre     : 1010000000 │
│ Contexto CPU   : PC=0x8A20 | SP=0xE000 | AC=0      │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 10         │
│ Memoria Asign. : 

## 3. Sección Práctica 2: Jerarquía de Procesos.  

En esta sección explorarás y comprenderás la jerarquía, el ciclo de vida y la gestión de procesos en sistemas operativos mediante la inspección de estructuras del kernel en Linux y la simulación algorítmica de llamadas al sistema (`fork`, `exec`, `exit`, `wait`)

### [CÓDIGO 3.1] Inspección de la Jerarquía de Procesos en Linux.

Ejecutar los siguientes comandos en celdas de código bash en Google Colab:
```bash
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state
```

In [4]:
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state

docker-init(1)─┬─node(7)─┬─colab-fileshim.(76)
               │         ├─jupyter-server(106)─┬─python3(3700)─┬─pstree(4121)
               │         │                     │               ├─{python3}(3704)
               │         │                     │               ├─{python3}(3705)
               │         │                     │               ├─{python3}(3706)
               │         │                     │               ├─{python3}(3707)
               │         │                     │               ├─{python3}(3708)
               │         │                     │               ├─{python3}(3709)
               │         │                     │               ├─{python3}(3710)
               │         │                     │               ├─{python3}(3711)
               │         │                     │               └─{python3}(3716)
               │         │                     ├─{jupyter-server}(124)
               │         │                     ├─{jupyter-server}(3689)
   

### [CÓDIGO 3.2] Inspección de la Jerarquía de Procesos en Linux.

Simulación de Jerarquía, BCP y Cambio de Contexto, copiar y ejecutar el siguiente código en Colab:

```cpp
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}
```

In [5]:
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <vector>
# include <string>
# include <iomanip>
# include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}

Writing laboratorio_jerarquia_bcp.cpp


### [CÓDIGO 3.3] Compilación y Ejecución del código
```c++
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp
```

In [6]:
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp


               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]

--- 1. Creación de Jerarquía con fork() ---
  [FORK] Proceso Padre PID 1 creó al Proceso Hijo PID 101 (bash)
  [FORK] Proceso Padre PID 101 creó al Proceso Hijo PID 102 (gcc)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=100 | Prog: bash00000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog: gcc000000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]

--- 2. Reemplazo de Imagen con exec() ---
  [EXEC] Proceso PID 102 reemplaza su imagen por 'cc1_compiler'

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo:

## 4. Sección Práctica 3: Inter Process Communication (IPC).  

En esta sección implementarás y analizarás los mecanismos de Comunicación entre Procesos (IPC) y sincronización concurrente en sistemas operativos mediante la inspección de recursos del kernel en Linux y la resolución del problema clásico del Productor-Consumidor usando semáforos POSIX y tuberías en C++17.

### [CÓDIGO 4.1] Inspección de Mecanismos de IPC en el Entorno Linux.

Ejecutar los siguientes comandos en celdas de código bash para inspeccionar las facilidades de IPC del kernel Linux:
```bash
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo
```

In [7]:
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo


------ Message Queues --------
key        msqid      owner      perms      used-bytes   messages    

------ Shared Memory Segments --------
key        shmid      owner      perms      bytes      nattch     status      

------ Semaphore Arrays --------
key        semid      owner      perms      nsems     

1048576
prw-r--r-- 1 root root 0 Nov 27 23:56 /tmp/mi_tubería_fifo


### [CÓDIGO 4.2] Simulación del Productor-Consumidor con Semáforos POSIX y Threads.

Copiar y ejecutar el siguiente código en Colab:
```cpp
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <semaphore.h>
#include <fcntl.h>
#include <unistd.h>
#include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}
```

In [8]:
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <thread>
# include <vector>
# include <chrono>
# include <semaphore.h>
# include <fcntl.h>
# include <unistd.h>
# include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}

Writing laboratorio_ipc_semaphoro.cpp


### [CÓDIGO 4.3] Compilación y Ejecución del código.

```bash
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro
```

In [9]:
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro

laboratorio_ipc_semaphoro.cpp: In function ‘void probar_tubería_pipe()’:
laboratorio_ipc_semaphoro.cpp:105:15: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  105 |         close(fd); // Cierra el extremo de lectura que no utiliza
      |               ^~
      |               |
      |               int*
In file included from laboratorio_ipc_semaphoro.cpp:13:
/usr/include/unistd.h:358:23: note:   initializing argument 1 of ‘int close(int)’
  358 | extern int close (int __fd);
      |                   ~~~~^~~~
laboratorio_ipc_semaphoro.cpp:116:14: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  116 |         read(fd, buffer_lectura, sizeof(buffer_lectura));
      |              ^~
      |              |
      |              int*
In file included from /usr/include/unistd.h:1217:
/usr/inc

## 6. Desafío Asíncrono / Entregable de la Unidad I

### Extensión y Análisis de la Tabla de Procesos y el BCP
A partir del laboratorio realizado en Google Colab sobre inspección de procesos en Linux y la estructura struct BCP en C++17, resuelve de manera individual los siguientes ítems modificando y extendiendo el código fuente base.

#### Instrucciones::
1. Análisis de Procesos del Sistema en Linux (/proc)
Contexto: El sistema de archivos virtual /proc contiene directorios nombrados con el PID de cada proceso activo, exponiendo archivos de control e información detallada del kernel.  
**Consigna:**  
> Ejecuta en una celda de Colab un comando o un pequeño script en Python/Bash que busque e imprima el uso de memoria actual y el estado de ejecución (campo State) de tres procesos distintos del sistema leyendo directamente sus archivos dentro de /proc/[PID]/status.  
Explica brevemente en un párrafo por qué el directorio /proc no ocupa espacio real en el disco duro de almacenamiento masivo.

2. Ampliación del Modelo de Datos del BCP en C++17
Contexto: Un sistema operativo real almacena más información de gestión en el Bloque de Control de Proceso, especialmente relacionada con la contabilidad de tiempo y la administración de archivos abiertos.  
**Consigna:**  
>Modifica el archivo bcp_struct.cpp para incorporar los siguientes campos dentro de la struct BCP:  
uint64_t tiempo_cpu_ms; (Tiempo total de CPU consumido en milisegundos).  
uint8_t archivos_abiertos; (Número de descriptores de archivos o sockets abiertos por el proceso).  
Actualiza el método mostrarBCP() para que despliegue formateados estos nuevos atributos en la ficha técnica de la consola.

3. Simulación de Transición de Estados
Contexto: Los procesos cambian dinámicamente de estado en función de los eventos del sistema y la planificación del kernel (Modelo de 5 Estados).  
**Consigna:**  
>Implementa una función en C++ dentro del programa que reciba por referencia un objeto BCP y un evento (por ejemplo, una interrupción por fin de E/S o una expiración de cuanto de tiempo/quantum), de modo que simule la transición de estado (por ejemplo, pasando de BLOQUEADO a LISTO) e imprima un mensaje descriptivo del cambio ocurrido.

### Gestión Avanzada y Ciclo de Vida de Procesos.  

#### Instrucciones:
A partir del laboratorio de jerarquía de procesos y simulación del BCP, resuelve de manera individual los siguientes ítems ampliando el código fuente de laboratorio_jerarquia_bcp.cpp.

1. Análisis de Procesos Zombie en Sistemas Linux Reales
Contexto: En los sistemas operativos tipo Unix, un proceso zombie ocurre cuando un hijo termina su ejecución, pero su padre aún no recolecta su estado de salida mediante la llamada wait().  
**Consigna:**  
> Investiga y redacta brevemente cómo se visualiza un proceso en estado zombie utilizando comandos de consola en Linux (por ejemplo, el indicador Z en ps aux o top).  
Responde en un párrafo de análisis: ¿Por qué el kernel de Linux mantiene el Bloque de Control de Proceso (BCP/Task Struct) en memoria para un proceso que ya ha terminado de ejecutarse y liberado su código en disco?  

2. Automatización de la Adopción de Huérfanos
Contexto: En el código base actual, la adopción de procesos huérfanos requiere invocar de manera explícita el método atender_huerfanos(). En un sistema operativo real, el kernel gestiona esto de forma automática cuando un proceso padre muere.  
**Consigna:**
> Modifica la lógica dentro de la clase SistemaOperativoSimulado para que, al ejecutar el método simular_exit(), el sistema detecte de forma automática si el proceso que termina tenía hijos activos y reasigne de inmediato su ppid al proceso init (PID 1), eliminando la necesidad de llamar al método de huérfanos por separado en el programa principal main.  

3. Extensión del Modelo de Estados (Estado Suspendido)
Contexto: Los modelos de procesos reales suelen incorporar más estados para manejar suspensiones temporales por intervención del usuario o del sistema (por ejemplo, mediante señales como SIGSTOP y SIGCONT).  
**Consigna:**  
> Agrega el estado SUSPENDIDO a la enumeración Estado.  
Implementa un nuevo método en el simulador llamado simular_suspension(int pid) que permita cambiar el estado de un proceso a SUSPENDIDO y guarde una traza en consola describiendo el cambio de contexto realizado.

### Concurrencia Avanzada y Robustez en IPC.  


A partir del laboratorio de mecanismos de IPC y sincronización, resuelve de manera individual los siguientes ítems ampliando y depurando el código fuente base.
#### Instrucciones:  

1. Auditoría de Recursos IPC en Linux.  
Contexto: Los recursos de IPC creados por los programas en sistemas Unix deben ser gestionados cuidadosamente por el kernel para evitar fugas de recursos (resource leaks).  
**Consigna:**  
> Ejecuta en una celda de Google Colab los comandos ipcs -m (para memoria compartida) e ipcs -s (para semáforos del sistema) y captura su salida.  
Explica brevemente en un párrafo qué representan las columnas clave (key), identificador (shmid/semid) y propietario (owner) que muestra el sistema operativo.

2. Escalabilidad a Múltiples Productores y Consumidores.  
Contexto: En escenarios reales de sistemas operativos, múltiples hilos o procesos intentan acceder concurrentemente a los recursos compartidos.  
**Consigna:**
> Modifica la función main() del código base para instanciar dos hilos productores y dos hilos consumidores de forma simultánea. Asegúrate de verificar que el semáforo de exclusión mutua (sem_mutex) y los contadores de huecos/elementos previenen eficazmente las condiciones de carrera y la corrupción del buffer circular.

3. Depuración y Análisis de Descriptores en Tuberías (Pipe).  
Contexto: La correcta gestión de los descriptores de archivos (file descriptors) es vital en las tuberías para evitar bloqueos permanentes (deadlocks) por lecturas colgadas en la espera de un fin de archivo (EOF).  
**Consigna:**  
> Analiza la función probar_tubería_pipe(). Identifica por qué el manejo de los descriptores de archivo requiere un arreglo de tamaño 2 (int fd[2]) para separar adecuadamente el canal de lectura (fd) del canal de escritura (fd), y documenta qué ocurre si el proceso padre olvida cerrar el extremo de escritura antes de realizar la operación de lectura bloqueante.
---

### PARTE 1: Extensión y Análisis de la Tabla de Procesos y el BCP

#### 1. Script en Python para leer el estado y memoria de 3 procesos en `/proc`

In [10]:
import os

def obtener_info_proceso(pid):
    ruta = f"/proc/{pid}/status"
    if not os.path.exists(ruta):
        return f"Proceso {pid} no encontrado."

    nombre, estado, memoria = "Desconocido", "Desconocido", "No reporta"
    try:
        with open(ruta, 'r') as f:
            for linea in f:
                if linea.startswith("Name:"):
                    nombre = linea.split()[1].strip()
                elif linea.startswith("State:"):
                    estado = " ".join(linea.split()[1:]).strip()
                elif linea.startswith("VmRSS:"):
                    memoria = " ".join(linea.split()[1:]).strip()
        return f"PID: {pid} | Nombre: {nombre} | Estado: {estado} | Memoria (RSS): {memoria}"
    except Exception as e:
        return f"Error al leer PID {pid}: {e}"

# Buscamos 3 PIDs activos en el sistema (ej. el actual, su padre, e init (1))
pid_actual = os.getpid()
pid_padre = os.getppid()
pids = [1, pid_padre, pid_actual]

print("=== INFORMACIÓN DE PROCESOS DESDE /PROC ===")
for pid in pids:
    print(obtener_info_proceso(pid))

=== INFORMACIÓN DE PROCESOS DESDE /PROC ===
PID: 1 | Nombre: docker-init | Estado: S (sleeping) | Memoria (RSS): 560 kB
PID: 106 | Nombre: jupyter-server | Estado: R (running) | Memoria (RSS): 162436 kB
PID: 3700 | Nombre: python3 | Estado: R (running) | Memoria (RSS): 105812 kB


**Análisis Teórico:**
¿Por qué el directorio `/proc` no ocupa espacio real en el disco duro?

El directorio `/proc` es un **sistema de archivos virtual** (o pseudo-sistema de archivos) provisto de forma directa por el kernel de Linux. No existe físicamente en el disco de almacenamiento masivo; en su lugar, el kernel genera la información en tiempo real directamente desde las estructuras de datos de la memoria RAM del sistema cuando un proceso solicita leer dichos archivos. Por ello, su tamaño reportado en disco siempre es de 0 bytes, ya que actúa como una ventana de visualización al estado interno del Kernel.

#### 2. Ampliación del Modelo de Datos del BCP en C++17 y Transición de Estados

Modificación del archivo `bcp_struct.cpp` para incorporar:
- Contabilidad de `tiempo_cpu_ms` y `archivos_abiertos`.
- Función de transición de estados por eventos.

In [11]:
%%writefile bcp_struct_ampliado.cpp
#include <iostream>
#include <string>
#include <vector>
#include <iomanip>
#include <cstdint>

enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

struct BCP {
    // 1. Identificadores
    uint32_t pid;
    uint32_t ppid;
    std::string nombre_programa;

    // 2. Información de Estado del Procesador
    EstadoProceso estado_actual;
    uint16_t PC;
    uint16_t SP;
    int AC;
    uint8_t PSW_modo; // 0 = Usuario, 1 = Kernel

    // 3. Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;
    uint32_t limite_memoria_bytes;
    int prioridad;

    // 4. NUEVOS CAMPOS ADICIONALES (Desafío)
    uint64_t tiempo_cpu_ms;
    uint8_t archivos_abiertos;

    void mostrarBCP() const {
        std::cout << "┌" << std::string(75, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(25) << nombre_programa << "│\n";
        std::cout << "├" << std::string(75, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(15) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(11) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(15) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(11) << limite_memoria_bytes << " Bytes   │\n";
        // Imprimir nuevos campos formateados
        std::cout << "│ Tiempo CPU     : " << std::setw(11) << tiempo_cpu_ms << " ms"
                  << "   | Files Abiertos  : " << std::setw(15) << (int)archivos_abiertos << " │\n";
        std::cout << "└" << std::string(75, '-') << "┘\n";
    }
};

// Función de simulación de transición de estados
void transicionarEstado(BCP& bcp, const std::string& evento) {
    std::cout << "[TRANSICIÓN] Proceso PID " << bcp.pid << " afectado por Evento: '" << evento << "'\n";
    EstadoProceso anterior = bcp.estado_actual;

    if (evento == "EXPIRACION_QUANTUM" && bcp.estado_actual == EstadoProceso::EJECUTANDO) {
        bcp.estado_actual = EstadoProceso::LISTO;
    } else if (evento == "ES_COMPLETADA" && bcp.estado_actual == EstadoProceso::BLOQUEADO) {
        bcp.estado_actual = EstadoProceso::LISTO;
    } else if (evento == "ES_SOLICITADA" && bcp.estado_actual == EstadoProceso::EJECUTANDO) {
        bcp.estado_actual = EstadoProceso::BLOQUEADO;
    } else if (evento == "PLANIFICADO" && bcp.estado_actual == EstadoProceso::LISTO) {
        bcp.estado_actual = EstadoProceso::EJECUTANDO;
    }

    std::cout << "             ↳ Estado anterior: " << estadoToString(anterior)
              << " -> Estado nuevo: " << estadoToString(bcp.estado_actual) << "\n\n";
}

int main() {
    BCP p1{101, 1, "chrome.exe", EstadoProceso::EJECUTANDO, 0x0420, 0x0FFF, 150, 0, 0x10000, 65536, 5, 4500, 4};
    p1.mostrarBCP();

    // Simular un cambio de estado por solicitud de E/S
    transicionarEstado(p1, "ES_SOLICITADA");
    p1.mostrarBCP();

    // Simular un cambio de estado por fin de E/S
    transicionarEstado(p1, "ES_COMPLETADA");
    p1.mostrarBCP();

    return 0;
}

Writing bcp_struct_ampliado.cpp


In [12]:
!g++ -O3 -std=c++17 bcp_struct_ampliado.cpp -o bcp_struct_ampliado
!./bcp_struct_ampliado

┌---------------------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   101) - Program: chrome.exe               │
├---------------------------------------------------------------------------┤
│ Estado Actual  : EJECUTANDO      | PPID Padre     : 1               │
│ Contexto CPU   : PC=0x4200 | SP=0xFFF0 | AC=150         │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 5               │
│ Memoria Asign. : Base=0x100000 | Límite=65536000000 Bytes   │
│ Tiempo CPU     : 45000000000 ms   | Files Abiertos  : 400000000000000 │
└---------------------------------------------------------------------------┘
[TRANSICIÓN] Proceso PID 101 afectado por Evento: 'ES_SOLICITADA'
             ↳ Estado anterior: EJECUTANDO -> Estado nuevo: BLOQUEADO

┌---------------------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID: 10100) - Program: chrome.exe000000000000000│
├-----------------------------------------------------------

### PARTE 2: Gestión Avanzada y Ciclo de Vida de Procesos

#### 1. Análisis de Procesos Zombie en Linux Reales
**¿Cómo identificar procesos zombies?**
En Linux, los procesos zombies se visualizan mediante comandos utilizando las siguientes banderas:
* En `ps aux` o `ps -ef`, aparecen marcados con el estado `Z` en la columna `STAT` (o `s`), y habitualmente acompañados del texto `<defunct>`.
* En la herramienta interactiva `top` o `htop`, la sección resumen de la parte superior muestra el recuento exacto bajo la etiqueta `zombie`.

**Análisis Teórico:**
¿Por qué se mantiene el BCP (Task Struct) en memoria para un proceso terminado?

El kernel mantiene la estructura del BCP en la tabla de procesos debido a que almacena información esencial que el proceso padre podría necesitar recolectar, como el código de salida (`exit status`), el ID de proceso (`PID`) y estadísticas de uso de recursos. Hasta que el padre no realice una llamada al sistema colectora (`wait()` o `waitpid()`), el kernel no puede remover de manera segura estos metadatos.

#### 2. Automatización de la Adopción de Huérfanos y Estado Suspendido en C++17
Ampliación de `laboratorio_jerarquia_bcp.cpp` con adopción automática en `simular_exit()` y el estado `SUSPENDIDO`.

In [13]:
%%writefile laboratorio_jerarquia_bcp_avanzado.cpp
#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <memory>
#include <algorithm>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO, SUSPENDIDO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        case Estado::SUSPENDIDO: return "SUSPENDIDO";
        default:                 return "DESCONOCIDO";
    }
}

struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC;
    uint16_t SP;
    int PSW_modo;
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(15) << nombre_programa
                  << " | Estado: " << std::setw(11) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;

public:
    SistemaOperativoSimulado() {
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
    }

    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo{contador_pid, padre->pid, nombre_hijo, Estado::LISTO, padre->PC, (uint16_t)(padre->SP - 0x0100), 0, 0};
        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de exit() con adopción automática de huérfanos incorporada
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE\n";

        // Automatización: Adoptar de inmediato hijos activos reasignándolos a init (PID 1)
        for (auto& p : tabla_procesos) {
            if (p.ppid == pid && p.estado != Estado::ZOMBIE) {
                std::cout << "  [AUTODETECCIÓN/ADOPCIÓN] El hijo huérfano PID " << p.pid
                          << " ('" << p.nombre_programa << "') fue reasignado automáticamente al proceso raíz init (PID 1)\n";
                p.ppid = 1;
            }
        }
    }

    // Simular suspensión de un proceso
    void simular_suspension(int pid) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->estado = Estado::SUSPENDIDO;
        std::cout << "  [SUSPENSIÓN] El proceso PID " << pid << " ('" << proc->nombre_programa
                  << "') ha sido suspendido (contexto preservado: PC=0x"
                  << std::hex << std::uppercase << proc->PC << std::dec << ")\n";
    }

    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid << ".\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;

    std::cout << "--- 1. Preparar Jerarquía ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Suspensión de un Proceso ---\n";
    so.simular_suspension(pid_worker);
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Muerte del Padre y Adopción Automática ---\n";
    so.simular_exit(pid_bash, 0);
    so.mostrarTablaProcesos();

    return 0;
}

Writing laboratorio_jerarquia_bcp_avanzado.cpp


In [14]:
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp_avanzado.cpp -o laboratorio_jerarquia_bcp_avanzado
!./laboratorio_jerarquia_bcp_avanzado

--- 1. Preparar Jerarquía ---
  [FORK] Proceso Padre PID 1 creó al Proceso Hijo PID 101 (bash)
  [FORK] Proceso Padre PID 101 creó al Proceso Hijo PID 102 (worker_task)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init            | Estado: EJECUTANDO  | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=100 | Prog: bash00000000000 | Estado: LISTO000000 | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog: worker_task0000 | Estado: LISTO000000 | PC: 0x1000 | Modo: USUARIO]

--- 2. Suspensión de un Proceso ---
  [SUSPENSIÓN] El proceso PID 102 ('worker_task') ha sido suspendido (contexto preservado: PC=0x10)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000000 | Estado: EJECUTANDO0 | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=100 | Prog: bash00000000000 | Estado: LISTO000000 | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog

### PARTE 3: Concurrencia Avanzada y Robustez en IPC

#### 1. Auditoría de Recursos IPC en Linux

In [15]:
# Capturando recursos de Memoria compartida y Semáforos activos
!ipcs -m
!ipcs -s


------ Shared Memory Segments --------
key        shmid      owner      perms      bytes      nattch     status      


------ Semaphore Arrays --------
key        semid      owner      perms      nsems     



**Análisis Teórico:**
* **`key`**: Es un valor numérico (generalmente expresado en hexadecimal) que sirve como identificador externo de IPC para que procesos independientes puedan localizar y asociarse al mismo segmento utilizando una clave común.
* **`shmid / semid`**: Es el identificador único interno que el kernel asigna a ese recurso una vez creado. Es el descriptor que se usa en las llamadas del sistema (`shmat`, `semop`).
* **`owner`**: Especifica el usuario (propietario) del sistema que posee los privilegios de acceso y control sobre el recurso IPC.

#### 2. Depuración del bug de Tuberías y Escalabilidad a Múltiples Productores/Consumidores

Aquí solucionamos el bug de compilación de descriptores `int fd[2]` y escalamos a **2 productores y 2 consumidores** concurrentes.

In [16]:
%%writefile laboratorio_ipc_final.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <semaphore.h>
#include <fcntl.h>
#include <unistd.h>
#include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS_POR_HEBRA = 5;

int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

sem_t sem_huecos;
sem_t sem_elementos;
sem_t sem_mutex;

void productor(int id) {
    for (int i = 1; i <= TOTAL_ITEMS_POR_HEBRA; ++i) {
        int dato = (id * 100) + i;

        sem_wait(&sem_huecos);
        sem_wait(&sem_mutex);

        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR " << id << "] Insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;

        sem_post(&sem_mutex);
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(100));
    }
}

void consumidor(int id) {
    for (int i = 1; i <= TOTAL_ITEMS_POR_HEBRA; ++i) {
        sem_wait(&sem_elementos);
        sem_wait(&sem_mutex);

        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR " << id << "] Extrajo item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;

        sem_post(&sem_mutex);
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (CORREGIDO)          \n";
    std::cout << "=================================================================\n";

    int fd[2]; // CORREGIDO: Arreglo de tamaño 2 necesario para entrada/salida
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();
    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // Proceso Hijo (Escritor)
        close(fd[0]); // Cierra lectura
        std::string mensaje = "Saludos del hijo a traves de Pipe POSIX!";
        std::cout << "  [HIJO] Enviando mensaje por fd[1] (extremo de escritura)...\n";
        write(fd[1], mensaje.c_str(), mensaje.length() + 1);
        close(fd[1]); // Cierra escritura (envía EOF)
        _exit(0);
    } else {
        // Proceso Padre (Lector)
        close(fd[1]); // Cierra escritura
        char buffer_lectura[100];
        std::cout << "  [PADRE] Esperando lectura en fd[0] (extremo de lectura)...\n";
        read(fd[0], buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PADRE] Mensaje recibido: \"" << buffer_lectura << "\"\n";
        close(fd[0]);
        wait(NULL);
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   PROD-CONS MULTI-HILO (2 PRODUCTORES - 2 CONSUMIDORES)         \n";
    std::cout << "=================================================================\n\n";

    sem_init(&sem_huecos, 0, TAMANO_BUFFER);
    sem_init(&sem_elementos, 0, 0);
    sem_init(&sem_mutex, 0, 1);

    // Lanzamiento de múltiples hilos concurrentes
    std::thread p1(productor, 1);
    std::thread p2(productor, 2);
    std::thread c1(consumidor, 1);
    std::thread c2(consumidor, 2);

    p1.join();
    p2.join();
    c1.join();
    c2.join();

    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    probar_tubería_pipe();
    return 0;
}

Writing laboratorio_ipc_final.cpp


In [17]:
!g++ -O3 -std=c++17 laboratorio_ipc_final.cpp -o laboratorio_ipc_final -pthread
!./laboratorio_ipc_final

laboratorio_ipc_final.cpp: In function ‘void probar_tubería_pipe()’:
laboratorio_ipc_final.cpp:79:14: warning: ignoring return value of ‘ssize_t write(int, const void*, size_t)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   79 |         write(fd[1], mensaje.c_str(), mensaje.length() + 1);
      |         ~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
laboratorio_ipc_final.cpp:87:13: warning: ignoring return value of ‘ssize_t read(int, void*, size_t)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   87 |         read(fd[0], buffer_lectura, sizeof(buffer_lectura));
      |         ~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
   PROD-CONS MULTI-HILO (2 PRODUCTORES - 2 CONSUMIDORES)         

  [PRODUCTOR 1] Insertó item 101 en la posición 0
  [PRODUCTOR 2] Insertó ite

#### 3. Depuración y Análisis de Descriptores en Tuberías (Pipe)
**Análisis Teórico:**
* **Necesidad de `fd[2]`:** Una tubería estándar es unidireccional. El kernel requiere mapear dos descriptores de archivos lógicos distintos en la tabla de descriptores de archivos del proceso: `fd[0]` configurado únicamente para operaciones de lectura bloqueante, y `fd[1]` configurado únicamente para la escritura.
* **Consecuencias de no cerrar el extremo de escritura en el Padre:** Si el proceso lector (Padre) olvida cerrar su copia del extremo de escritura (`fd[1]`), el descriptor de escritura se mantiene abierto en el sistema. Al intentar realizar la lectura, la función `read()` nunca recibirá la señal de fin de archivo (`EOF`) aun cuando el hijo finalice, provocando un bloqueo permanente (**deadlock**) al quedar el lector colgado indefinidamente esperando datos de un canal que sigue potencialmente abierto.

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica_Sistemas Operativos U2.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.